In [1]:
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
import pymaster as nmt
root_dir='/home/matteo/Scrivania/workspace/'
Cls_dir=root_dir+'simulazione/'
ws_dir=root_dir+'simulazione/'+'ws/'
mask_dir=root_dir+'simulazione/'+'mask/'
maps_dir=root_dir+'simulazione/'+'maps/'
Cls_out_dir=root_dir+'simulazione/'+'spectra_out/'
newmaps=False #se True, genera nuove mappe altrimenti carica quelle esistenti
real_mask=False #se True, importa una maschera altrimenti genera una maschera identità
purify_e=False
purify_b=False
nside=64
nsims=1000
deltal=5
lmax=2*nside #usare 3nside-1 se si usa purify_e e purify_b
sigma=2.0
Spix=hp.nside2pixarea(nside, degrees=True)
var=sigma*sigma/(Spix*3600.) 
npix=hp.nside2npix(nside)
fwhm=2.4*np.sqrt(Spix)
ell=np.arange(0,lmax+1) 
ll=ell*(ell+1)/2/np.pi

In [2]:

purify_e=False
purify_b=False
    
b = nmt.NmtBin.from_lmax_linear(lmax, deltal, is_Dell=False)
ell_arr = b.get_effective_ells()
bl = hp.gauss_beam(np.deg2rad(fwhm/60.0), lmax=lmax, pol=True)
wpix = hp.pixwin(nside, pol=True, lmax=lmax)

TT_th,EE_th,BB_th,TE_th=np.loadtxt(Cls_dir+'ffp10_lensedCls.dat',unpack=True,usecols=(1,2,3,4))
Cls_th=np.zeros((4,lmax+1))
Cls_th[0,2:]=TT_th[0:lmax-1]/ll[2:]
Cls_th[1,2:]=EE_th[0:lmax-1]/ll[2:]
Cls_th[2,2:]=BB_th[0:lmax-1]/ll[2:]
Cls_th[3,2:]=TE_th[0:lmax-1]/ll[2:]

noisesT = np.random.normal(loc=0,scale=np.sqrt(var),size=2*npix)
noisesP = np.random.normal(loc=0,scale=np.sqrt(2*var),size=4*npix)
m_sign=hp.synfast(cls=Cls_th ,lmax=lmax,nside=nside,pol=True,new=True,fwhm=np.deg2rad(fwhm/60.0), pixwin=True)
ma_TQU=np.zeros_like(m_sign)
mb_TQU=np.zeros_like(m_sign)

ma_TQU[0]=m_sign[0]+noisesT[0:npix]
ma_TQU[1]=m_sign[1]+noisesP[0:npix]
ma_TQU[2]=m_sign[2]+noisesP[npix:2*npix]
mb_TQU[0]=m_sign[0]+noisesT[npix:2*npix]
mb_TQU[1]=m_sign[1]+noisesP[2*npix:3*npix]
mb_TQU[2]=m_sign[2]+noisesP[3*npix:4*npix]
  




In [3]:
maskT_in = hp.read_map(mask_dir+'COM_Mask_CMB-common-Mask-Int_2048_R3.00.fits')
maskP_in = hp.read_map(mask_dir+'COM_Mask_CMB-common-Mask-Pol_2048_R3.00.fits')
downgradeT=hp.ud_grade(maskT_in, nside)
downgradeP=hp.ud_grade(maskP_in, nside)
aposize=1.05
maskT_out = nmt.mask_apodization(downgradeT,aposize,apotype="C1")
maskP_out = nmt.mask_apodization(downgradeP,aposize,apotype="C1")  

In [7]:
hp.nside2resol(nside=256,arcmin=True)

np.float64(13.741945647269624)

0.0005817764173314432 0.0005817764173314432
